# Chatbot RAG — The Acts of the Apostles

## 1. Indexação dos embeddings com FAISS

Etapas:
1. Ler o `.txt` e separar o livro por capítulo (cada capítulo começa com `\fChap. N - Título`).
2. Quebrar cada capítulo em *chunks* menores.
3. Gerar os embeddings com `sentence-transformers/all-MiniLM-L6-v2`.
4. Indexar no FAISS e salvar o índice em disco.

In [2]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

SOURCE_FILE = Path("rag-file/The-Acts-of-the-Apostles.txt")
INDEX_DIR = Path("faiss-index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

/var/folders/bz/0v3wc0m9463b0x7_226b89cm0000gn/T/ipykernel_34866/3110406028.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 1.1 Carregar o texto e separar por capítulo

No corpo do livro, cada capítulo começa com um *form feed* (`\f`) seguido de `Chap. N - Título`. O título pode ocupar duas linhas.
Tudo o que vem antes do capítulo 1 (licença, prefácio, sumário) fica de fora.
Cada quebra de página traz o número da página numa linha própria, seguido de `\f`. Esses números são removidos para não aparecerem no meio das frases.
O texto também tem quebras de linha no meio das frases, então os espaços em branco são normalizados.

In [3]:
text = SOURCE_FILE.read_text(encoding="utf-8")

PAGE_BREAK_RE = re.compile(r"\n\s*\d{1,3}\s*(?:\f|$)")
CHAPTER_RE = re.compile(r"\fChap\.\s*(\d+)\s*-\s*([^\n]+(?:\n[^\n]+)?)\n\n")
matches = list(CHAPTER_RE.finditer(text))

chapters = []
for i, m in enumerate(matches):
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    body = PAGE_BREAK_RE.sub(" ", text[m.end():end])
    body = re.sub(r"\s+", " ", body).strip()
    chapters.append(Document(
        page_content=body,
        metadata={
            "chapter": int(m.group(1)),
            "title": re.sub(r"\s+", " ", m.group(2)).strip(),
            "source": SOURCE_FILE.name,
        },
    ))

print(f"{len(chapters)} capítulos")
chapters[0].metadata, chapters[0].page_content[:300]

58 capítulos


({'chapter': 1,
  'title': 'God’s Purpose for His Church',
  'source': 'The-Acts-of-the-Apostles.txt'},
 'The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the chu')

### 1.2 Dividir em chunks

O `all-MiniLM-L6-v2` trunca a entrada em 256 *tokens*, ou seja, uns 1.000 caracteres em inglês.
Chunks de ~800 caracteres com 100 de sobreposição ficam abaixo desse limite.
Cada chunk herda o capítulo e o título como metadados.

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[". ", "? ", "! ", "; ", ", ", " ", ""],
    keep_separator="end",
)
chunks = splitter.split_documents(chapters)

print(f"{len(chunks)} chunks")
chunks[0]

1317 chunks


Document(metadata={'chapter': 1, 'title': 'God’s Purpose for His Church', 'source': 'The-Acts-of-the-Apostles.txt'}, page_content='The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the church, those whom He has called out of darkness into His marvelous light, are to show forth His glory. The church is the repository of the riches of the grace of Christ; and through the church will eventually be made manifest, even to “the principalities and powers in heavenly places,” the final and full display of the love of God. Ephesians 3:10. Many and wonderful are the promises recorded in the Scriptures regarding the church.')

### 1.3 Gerar os embeddings e indexar no FAISS

Com `normalize_embeddings=True`, ordenar pela distância L2 dá o mesmo resultado que ordenar pela similaridade de cosseno.

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(str(INDEX_DIR))

print(f"{vectorstore.index.ntotal} vetores de dimensão {vectorstore.index.d} salvos em {INDEX_DIR}/")

/Users/eduar/.pyenv/versions/acts-of-apostle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 16:46:21.599000 34866 /Users/eduar/.pyenv/versions/3.12.9/envs/acts-of-apostle/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12133.50it/s]


1317 vetores de dimensão 384 salvos em faiss-index/


### 1.4 Testar o retrieval

Depois de salvo, o índice pode ser recarregado sem gerar os embeddings de novo.
`allow_dangerous_deserialization` é necessário porque o docstore é salvo em pickle; só use com índices gerados por você.

In [14]:
vectorstore = FAISS.load_local(
    str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
)

query = "What happened with Stephen?"
for doc, score in vectorstore.similarity_search_with_score(query, k=4):
    print(f"[{score:.3f}] Cap. {doc.metadata['chapter']} - {doc.metadata['title']}")
    print(doc.page_content[:250], "...\n")

[0.842] Cap. 10 - The First Christian Martyr
The martyrdom of Stephen made a deep impression upon all who witnessed it. The memory of the signet of God upon his face; his words, which touched the very souls of those who heard them, remained in the minds of the beholders, and testified to the tr ...

[0.848] Cap. 13 - Days of Preparation
Although “he spake boldly in the name of the Lord Jesus, and disputed against the Grecians,” those who stood at the head of the Jewish church refused to believe, but “went about to slay him.” Sorrow filled his heart. He would willingly have yielded u ...

[0.862] Cap. 10 - The First Christian Martyr
He connected Jesus Christ with all the Jewish history. He referred to the building of the temple by Solomon, and to the words of both Solomon and Isaiah: “Howbeit the Most High dwelleth not in temples made with hands; as saith the prophet, Heaven is  ...

[0.952] Cap. 10 - The First Christian Martyr
Stopping their ears, that they might not hear his words, an